In [1]:
import pandas as pd
import glob
import os
import re
import random

In [2]:
!pwd

/home/ylnner/Documents/omnet++/omnetpp-6.3.0/projects/florasat_v4_ult_routing/simulations/Test_Individual


# Functions

In [3]:
def process_files(files):

    dfs = []
    for file in files:
        # Leer el archivo csv
        df = pd.read_csv(file)

        #print("file: ", file)
        # 3. Extraer 'alt' y 'raan' del nombre del archivo usando expresiones regulares
        alt_match = re.search(r'alt_(\d+)', file)
        raan_match = re.search(r'raan_(\d+)', file)

        #df['alt'] = alt_match
        #df['raan'] = raan_match


        if alt_match:
            df['alt'] = int(alt_match.group(1))
        if raan_match:
            df['raan'] = int(raan_match.group(1))

        df['id_simulation'] = 'sim_alt_' +str(int(alt_match.group(1)))+ '_raan_' + str(int(raan_match.group(1)))

        #''' 
        print(df.shape)
        print('id_simulation: ', 'sim_alt_' +str(int(alt_match.group(1)))+ '_raan_' + str(int(raan_match.group(1))))
        print('0 : ', df[df['rcvOk'] == 0].shape[0], ' | ' )#,round(df[df['rcvOk'] == 0].shape[0] * 100 / df.shape[0],2))
        print('1 : ', df[df['rcvOk'] == 1].shape[0], ' | ' )#,round(df[df['rcvOk'] == 1].shape[0] * 100 / df.shape[0],2))
        #'''
        dfs.append(df)

    # 4. Concatenar todos los DataFrames
    if dfs:
        final_df = pd.concat(dfs, ignore_index=True)

        # Guardar el resultado final en un nuevo archivo
        #output_filename = "Concatenated_TransmissionStatistics.csv"
        #final_df.to_csv(output_filename, index=False)

        #print(f"Se concatenaron {len(files)} archivos con éxito.")
        #print(final_df.head())
    else:
        print("No se encontraron archivos que coincidan con el patrón.")

    
    #final_df.to_csv('final_data_tranmissions.csv', index=False)
    return final_df

# Seed

In [4]:
base_path = os.path.join("results", "repetitions")

# Getting seed folders
seed_folders = sorted(glob.glob(os.path.join(base_path, "seed_*/")))
seed_folders

['results/repetitions/seed_0/',
 'results/repetitions/seed_1/',
 'results/repetitions/seed_2/',
 'results/repetitions/seed_3/',
 'results/repetitions/seed_4/',
 'results/repetitions/seed_5/',
 'results/repetitions/seed_6/',
 'results/repetitions/seed_7/',
 'results/repetitions/seed_8/',
 'results/repetitions/seed_9/']

In [5]:
dict_train_files = {}
dict_test_files = {}


dict_test_files[600] = [200]
dict_test_files[850] = [330]
dict_test_files[740] = [150, 180]
dict_test_files[700] = [190]
dict_test_files

{600: [200], 850: [330], 740: [150, 180], 700: [190]}

In [7]:
# Ruta base donde están las carpetas seed_X
base_path = os.path.join("results", "repetitions")

# Getting seed folders
seed_folders = sorted(glob.glob(os.path.join(base_path, "seed_*/")))


all_train_pathfiles = []
all_test_pathfiles = []

seed_folders = sorted(glob.glob(os.path.join(base_path, "seed_*/")))
print('seed_folders: ', seed_folders)
#seed_folders

for seed_folder in seed_folders:
    print(f"--- Procesando: {(seed_folder)} ---")
    
    file_pattern = os.path.join(seed_folder, "TransmissionStatistics_*.csv")
    files_in_seed = glob.glob(file_pattern)

    print('files_in_seed: ', len(files_in_seed))
    train_pathfiles = []
    test_pathfiles  = []

    alt_identified = []
    for file in files_in_seed:
        alt_match = re.search(r'alt_(\d+)', file)
        raan_match = re.search(r'raan_(\d+)', file)
        alt_identified.append(int(alt_match.group(1)))

    alt_identified = set(alt_identified)
    print(alt_identified)

    new_file_pattern = os.path.join(seed_folder, "TransmissionStatistics_alt*.csv")

    for alt in alt_identified:
        print(">>alt: ", alt)
        aux = "TransmissionStatistics_alt_" + str(alt) + "*.csv"
        new_file_pattern = os.path.join(seed_folder, aux)
        new_files = glob.glob(new_file_pattern)
        
        ###num_files_train = int(len(new_files)*0.8)
        ###num_files_test  = len(new_files) - num_files_train        
        
        ###train_files_selected = random.sample(new_files, num_files_train)
        ###test_files_selected = [f for f in new_files if f not in train_files_selected]

        train_files_selected = []
        test_files_selected = []

        for file in new_files:
            #alt_match = re.search(r'alt_(\d+)', file)
            raan_match = re.search(r'raan_(\d+)', file)
            raan_identified = int(raan_match.group(1))
            #print('raan_identified: ',raan_identified)
            if alt in dict_test_files.keys():
                if raan_identified in dict_test_files[alt]:
                    test_files_selected.append(file)
                else:
                    train_files_selected.append(file)

        train_pathfiles.extend(train_files_selected)
        test_pathfiles.extend(test_files_selected)
        
    print(">test_files_selected: ", len(test_pathfiles), test_pathfiles)
    print(">train_pathfiles: ", len(train_pathfiles), train_pathfiles)

    #''' 
    train_csv = process_files(train_pathfiles)
    test_csv  = process_files(test_pathfiles)

    all_test_pathfiles.extend(test_pathfiles)
    all_train_pathfiles.extend(train_pathfiles)

    train_csv.to_csv(seed_folder + '_final_train_data_transmission.csv', index=False)    
    df_sorted = test_csv.sort_values(by=['id_simulation', 'time']).reset_index(drop=True)
    
    split_point = len(df_sorted) // 2
    
    test_df = df_sorted.iloc[:split_point].sort_values(by=['id_simulation', 'time']).reset_index(drop=True)
    val_df = df_sorted.iloc[split_point:].sort_values(by=['id_simulation', 'time']).reset_index(drop=True)

    test_df.to_csv(seed_folder + '_final_test_data_transmission.csv', index=False)
    val_df.to_csv(seed_folder + '_final_val_data_transmission.csv', index=False)

    print(f"Total: {len(df_sorted)} | Test: {len(test_df)} | Val: {len(val_df)}")
    #'''

seed_folders:  ['results/repetitions/seed_0/', 'results/repetitions/seed_1/', 'results/repetitions/seed_2/', 'results/repetitions/seed_3/', 'results/repetitions/seed_4/', 'results/repetitions/seed_5/', 'results/repetitions/seed_6/', 'results/repetitions/seed_7/', 'results/repetitions/seed_8/', 'results/repetitions/seed_9/']
--- Procesando: results/repetitions/seed_0/ ---
files_in_seed:  21
{600, 850, 700, 740}
>>alt:  600
>>alt:  850
>>alt:  700
>>alt:  740
>test_files_selected:  5 ['results/repetitions/seed_0/TransmissionStatistics_alt_600_raan_200_seed_0.csv', 'results/repetitions/seed_0/TransmissionStatistics_alt_850_raan_330_seed_0.csv', 'results/repetitions/seed_0/TransmissionStatistics_alt_700_raan_190_seed_0.csv', 'results/repetitions/seed_0/TransmissionStatistics_alt_740_raan_150_seed_0.csv', 'results/repetitions/seed_0/TransmissionStatistics_alt_740_raan_180_seed_0.csv']
>train_pathfiles:  16 ['results/repetitions/seed_0/TransmissionStatistics_alt_600_raan_170_seed_0.csv', 're

In [ ]:
test_pathfiles

In [ ]:
all_test_pathfiles

In [ ]:
/home/ylnner/Documents/omnet++/omnetpp-6.3.0/projects/florasat_v4_ult_routing/simulations/Test_Individual/results/repetitions

In [10]:
!pwd

/home/ylnner/Documents/omnet++/omnetpp-6.3.0/projects/florasat_v4_ult_routing/simulations/Test_Individual


In [11]:
import os
import glob
import pandas as pd


ruta_directorio = "results/repetitions/"
conjuntos = ["train", "val", "test"]

print("Iniciando el análisis de balance de clases...\n")

for conjunto in conjuntos:
    # Buscamos todos los archivos correspondientes al conjunto actual (ej. seed_0, seed_1, etc.)
    patron = os.path.join(ruta_directorio, f"seed_*_final_{conjunto}_data_transmission.csv")
    archivos = glob.glob(patron)
    
    if not archivos:
        print(f"No se encontraron archivos para el conjunto: {conjunto}")
        print("-" * 50)
        continue
    
    print(f"Se encontraron {len(archivos)} archivos para el conjunto '{conjunto}'.")
    
    lista_rcv_ok = []
    for archivo in archivos:
        try:
            # Cargamos únicamente la columna 'rcvOk' para optimizar la velocidad y el uso de memoria
            df = pd.read_csv(archivo, usecols=['rcvOk'])
            lista_rcv_ok.append(df['rcvOk'])
        except ValueError:
            print(f"Advertencia: La columna 'rcvOk' no se encuentra en {archivo}")
        except Exception as e:
            print(f"Error al leer el archivo {archivo}: {e}")
            
    if lista_rcv_ok:
        # Concatenamos todos los datos recopilados de las distintas repeticiones (seeds)
        datos_totales = pd.concat(lista_rcv_ok, ignore_index=True)
        
        # Calculamos la frecuencia absoluta y el porcentaje (frecuencia relativa)
        conteos = datos_totales.value_counts()
        porcentajes = datos_totales.value_counts(normalize=True) * 100
        
        # Creamos un DataFrame para mostrar el resumen de forma ordenada
        resumen = pd.DataFrame({
            'Cantidad': conteos,
            'Porcentaje (%)': porcentajes
        })
        
        # Ordenamos el índice para que muestre primero la clase 0 y luego la 1
        resumen = resumen.sort_index()
        
        print(f"\nBalance de clases para {conjunto.upper()}:")
        print(resumen.to_string())
        print("-" * 50)
    else:
        print(f"No se encontraron datos válidos para el conjunto '{conjunto}'.")
        print("-" * 50)

Iniciando el análisis de balance de clases...

Se encontraron 10 archivos para el conjunto 'train'.

Balance de clases para TRAIN:
       Cantidad  Porcentaje (%)
rcvOk                          
0        405415       80.439484
1         98585       19.560516
--------------------------------------------------
Se encontraron 10 archivos para el conjunto 'val'.

Balance de clases para VAL:
       Cantidad  Porcentaje (%)
rcvOk                          
0         63386       78.254321
1         17614       21.745679
--------------------------------------------------
Se encontraron 10 archivos para el conjunto 'test'.

Balance de clases para TEST:
       Cantidad  Porcentaje (%)
rcvOk                          
0         65664       81.066667
1         15336       18.933333
--------------------------------------------------


In [9]:
import os
import glob
import pandas as pd


#ruta_directorio = "results/repetitions/"
ruta_directorio = "results/repetitions_old_vpaper/"
conjuntos = ["train", "val", "test"]

print("Iniciando el análisis de balance de clases...\n")

for conjunto in conjuntos:
    # Buscamos todos los archivos correspondientes al conjunto actual (ej. seed_0, seed_1, etc.)
    patron = os.path.join(ruta_directorio, f"seed_*_final_{conjunto}_data_transmission.csv")
    archivos = glob.glob(patron)
    
    if not archivos:
        print(f"No se encontraron archivos para el conjunto: {conjunto}")
        print("-" * 50)
        continue
    
    print(f"Se encontraron {len(archivos)} archivos para el conjunto '{conjunto}'.")
    
    lista_rcv_ok = []
    for archivo in archivos:
        try:
            # Cargamos únicamente la columna 'rcvOk' para optimizar la velocidad y el uso de memoria
            df = pd.read_csv(archivo, usecols=['rcvOk'])
            lista_rcv_ok.append(df['rcvOk'])
        except ValueError:
            print(f"Advertencia: La columna 'rcvOk' no se encuentra en {archivo}")
        except Exception as e:
            print(f"Error al leer el archivo {archivo}: {e}")
            
    if lista_rcv_ok:
        # Concatenamos todos los datos recopilados de las distintas repeticiones (seeds)
        datos_totales = pd.concat(lista_rcv_ok, ignore_index=True)
        
        # Calculamos la frecuencia absoluta y el porcentaje (frecuencia relativa)
        conteos = datos_totales.value_counts()
        porcentajes = datos_totales.value_counts(normalize=True) * 100
        
        # Creamos un DataFrame para mostrar el resumen de forma ordenada
        resumen = pd.DataFrame({
            'Cantidad': conteos,
            'Porcentaje (%)': porcentajes
        })
        
        # Ordenamos el índice para que muestre primero la clase 0 y luego la 1
        resumen = resumen.sort_index()
        
        print(f"\nBalance de clases para {conjunto.upper()}:")
        print(resumen.to_string())
        print("-" * 50)
    else:
        print(f"No se encontraron datos válidos para el conjunto '{conjunto}'.")
        print("-" * 50)

Iniciando el análisis de balance de clases...

Se encontraron 10 archivos para el conjunto 'train'.

Balance de clases para TRAIN:
       Cantidad  Porcentaje (%)
rcvOk                          
0         38997       54.728023
1         32259       45.271977
--------------------------------------------------
Se encontraron 10 archivos para el conjunto 'val'.

Balance de clases para VAL:
       Cantidad  Porcentaje (%)
rcvOk                          
0          6823       59.007178
1          4740       40.992822
--------------------------------------------------
Se encontraron 10 archivos para el conjunto 'test'.

Balance de clases para TEST:
       Cantidad  Porcentaje (%)
rcvOk                          
0          6071       52.526389
1          5487       47.473611
--------------------------------------------------


In [17]:
!pwd

/home/ylnner/Documents/omnet++/omnetpp-6.3.0/projects/florasat_v4_ult_routing/simulations/Test_Individual


In [ ]:
df = pd.read_csv('results/repetitions/seed_0_final_train_data_transmission.csv')
df.shape

In [ ]:
old_df = pd.read_csv('results/repetitions_old_vpaper/seed_0_final_train_data_transmission.csv')
old_df.shape